In [46]:
import requests
import pandas as pd

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "PRECTOTCORR,T2M",
    "community": "AG",
    "latitude": -31.6,
    "longitude": -60.7,
    "start": "20000101",
    "end": "20241231",
    "format": "JSON",
}


resp = requests.get(url, params=params)
print("Codigo:", resp.status_code)
print("Llaves de la respuesta:", list(resp.json().keys()))
resp.json()["properties"]["parameter"].keys()

Codigo: 200
Llaves de la respuesta: ['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times']


dict_keys(['PRECTOTCORR', 'T2M'])

In [47]:
datos = resp.json()["properties"]["parameter"]
clima = pd.DataFrame(datos)

print("Forma:", clima.shape)
clima.head()

Forma: (9132, 2)


,PRECTOTCORR,T2M
20000101,0.0,26.75
20000102,0.0,26.43
20000103,0.0,26.97
20000104,0.0,28.08
20000105,0.0,29.85


In [48]:
#Sumamos por anio
clima["anio"] = clima.index.str[:4]

anual = clima.groupby("anio").agg(
    precip_total_mm=("PRECTOTCORR", "sum"),
    temp_media=("T2M", "mean"),
)

anual

,precip_total_mm,temp_media
anio,,
2000,1708.16,18.361230
2001,1188.33,19.304849
2002,1403.44,19.067863
2003,1319.53,18.659397
2004,976.35,19.589454
2005,1305.35,18.759397
2006,1060.29,19.870932
2007,1277.50,17.924986
2008,613.26,20.076585


In [49]:
#Ya comprobe que los resultados de una sola provincia son acordes. ahora intentemos traer los datos de todas las provincias

def clima_anual(lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "PRECTOTCORR,T2M,T2M_MAX,ALLSKY_SFC_SW_DWN,RH2M,WS2M,GWETROOT",
        "community": "AG",
        "latitude": lat,
        "longitude": lon,
        "start": "20000101",
        "end": "20241231",
        "format": "JSON",
    }
    resp = requests.get(url, params=params)
    datos = resp.json()["properties"]["parameter"]
    clima = pd.DataFrame(datos)
    clima["anio"] = clima.index.str[:4]
    anual = clima.groupby("anio").agg(
        precip_total_mm=("PRECTOTCORR", "sum"),
        temp_media=("T2M", "mean"),
        temp_max_media=("T2M_MAX", "mean"),
        radiacion_solar=("ALLSKY_SFC_SW_DWN", "mean"),
        humedad_rel=("RH2M", "mean"),
        viento=("WS2M", "mean"),
        humedad_suelo=("GWETROOT", "mean"),
    ).reset_index()
    return anual

In [50]:
#Lo probamos

prueba = clima_anual(-31.6, -60.7)
prueba.head()

,anio,precip_total_mm,temp_media,temp_max_media,radiacion_solar,humedad_rel,viento,humedad_suelo
0,2000,1708.16,18.361230,24.354317,17.492896,72.830492,2.632678,0.714645
1,2001,1188.33,19.304849,25.205589,17.101616,74.420329,2.693068,0.679726
2,2002,1403.44,19.067863,25.287288,17.161342,72.825726,2.776877,0.670712
3,2003,1319.53,18.659397,24.835425,17.643342,72.600219,2.641808,0.706082
4,2004,976.35,19.589454,26.230765,18.265601,66.484563,2.722951,0.652650


In [51]:
#Traemos el anterior dataset para ver que provincias son relevantes para extraerles el clima historico

import sqlite3

conn = sqlite3.connect("agro.db")
prod_prov = pd.read_sql_query("""
    SELECT provincia, SUM (produccion_tm) AS total
    FROM estimaciones
    GROUP BY provincia
    ORDER BY total DESC
""", conn)
conn.close()
prod_prov

,provincia,total
0,Buenos Aires,1771247388
1,Córdoba,1243207668
2,Santa Fe,922329946
3,Tucumán,334127389
4,Entre Ríos,293124834
5,Santiago del Estero,212057265
6,La Pampa,166896135
7,Salta,150508390
8,Chaco,130202596
9,Jujuy,114273283


In [52]:
# vamos a dar las coordenadas que me parecen bien de referencia para traer, algunas provincias con mas de una coordenada representativa para sacar promedio como cordoba zona marcos juares y san francisco

provincias_coords = {
    "Buenos Aires":        [(-34.0, -61.0), (-38.0, -59.5), (-36.0, -62.7)],
    "Córdoba":             [(-32.7, -62.1), (-33.1, -64.3), (-31.4, -62.1)],
    "Santa Fe":            [(-33.7, -61.9), (-31.3, -61.5)],
    "Tucumán":             [(-26.9, -65.0)],
    "Entre Ríos":          [(-32.0, -59.5), (-31.5, -58.5)],
    "Santiago del Estero": [(-28.0, -62.8), (-27.5, -64.3)],
    "La Pampa":            [(-35.5, -63.8), (-36.6, -64.3)],
    "Salta":               [(-24.8, -64.0), (-25.3, -64.2)],
    "Chaco":               [(-27.2, -61.2), (-26.8, -60.5)],
    "Jujuy":               [(-23.8, -64.7)],
    "San Luis":            [(-33.3, -65.3), (-34.0, -65.5)],
    "Misiones":            [(-26.8, -54.5), (-27.4, -55.1)],
    "Corrientes":          [(-29.0, -58.0), (-28.0, -56.5)],
}

In [53]:
tablas = []
for provincia, puntos in provincias_coords.items():
    print("Extrayendo clima de:", provincia, f"({len(puntos)} puntos)")
    sub = []
    for (lat, lon) in puntos:
        sub.append(clima_anual(lat, lon))
    t = (pd.concat(sub)
            .groupby("anio")
            .agg(precip_total_mm=("precip_total_mm", "mean"),
                temp_media=("temp_media", "mean"))
            .reset_index())
    t["provincia"] = provincia
    tablas.append(t)

clima_provincias = pd.concat(tablas, ignore_index=True)
print("\nForma final:", clima_provincias.shape)
clima_provincias.head()

Extrayendo clima de: Buenos Aires (3 puntos)
Extrayendo clima de: Córdoba (3 puntos)
Extrayendo clima de: Santa Fe (2 puntos)
Extrayendo clima de: Tucumán (1 puntos)
Extrayendo clima de: Entre Ríos (2 puntos)
Extrayendo clima de: Santiago del Estero (2 puntos)
Extrayendo clima de: La Pampa (2 puntos)
Extrayendo clima de: Salta (2 puntos)
Extrayendo clima de: Chaco (2 puntos)
Extrayendo clima de: Jujuy (1 puntos)
Extrayendo clima de: San Luis (2 puntos)
Extrayendo clima de: Misiones (2 puntos)
Extrayendo clima de: Corrientes (2 puntos)

Forma final: (325, 4)


,anio,precip_total_mm,temp_media,provincia
0,2000,1148.693333,14.804672,Buenos Aires
1,2001,1380.553333,15.254621,Buenos Aires
2,2002,1202.000000,14.870868,Buenos Aires
3,2003,824.900000,15.334685,Buenos Aires
4,2004,842.913333,15.866175,Buenos Aires


In [54]:
#Vamos con el join pero primero igualamos las tablas y guardo en la misma db

clima_provincias["anio"] = clima_provincias["anio"].astype(int)

conn = sqlite3.connect("agro.db")
clima_provincias.to_sql("clima", conn, if_exists="replace", index=False)
conn.close()

print("tabla 'clima' guardada en agro.db")

tabla 'clima' guardada en agro.db


In [55]:
conn = sqlite3.connect("agro.db")
test = pd.read_sql_query("""
    SELECT e.anio,
           ROUND(AVG(e.rendimiento_kgxha)) AS rinde_soja,
           ROUND(c.precip_total_mm) AS lluvia_mm
    FROM estimaciones e
    JOIN clima c ON e.provincia = c.provincia AND e.anio = c.anio
    WHERE e.cultivo = 'soja total' AND e.provincia = 'Santa Fe'
    GROUP BY e.anio
    ORDER BY e.anio
""", conn)
conn.close()
test

,anio,rinde_soja,lluvia_mm
0,2000,2551.0,1416.0
1,2001,2366.0,1212.0
2,2002,2809.0,1189.0
3,2003,2408.0,1038.0
4,2004,2569.0,971.0
5,2005,2528.0,1035.0
6,2006,3016.0,1062.0
7,2007,2751.0,1101.0
8,2008,1783.0,604.0
9,2009,3163.0,1109.0
